# Langfuse v3 — Dataset & Run Explorer / Exporter (REST · Judge · Circulars)

Explore datasets, runs, run items, **LLM-as-a-judge scores + reasoning + judge input prompt**,
and **circular metadata** pulled from the agent's retriever step, then export to a **styled,
read-ready Excel** plus flat + nested **JSON** — with full field control.

Talks to the Langfuse **Public REST API** directly via `httpx` (no SDK, no OTel) so the
self-signed / TLS-proxy SSL issues don't apply.

---
### What this extracts (quick map)
- **Dataset / item / run** metadata
- **Run items** joined to their main trace: app `input`, `actual_output`, dataset `expected_output`
- **Judge** (per evaluator): score `value`, `reasoning` (score.comment), the judge's own
  `input` prompt + raw `output` (from the judge's execution trace)
- **Circulars**: from the observation named *"Vector Store Retriever (Detailed)"*, parse
  `output.documents[*]` → `timestamp`, `filename`, `attachment_id`, `summary`

Full column definitions are written to a **README** sheet inside the Excel file.

## 1. Connection (REST client)

In [ ]:
import httpx

LANGFUSE_HOST       = "https://your-langfuse.internal"   # no trailing slash
LANGFUSE_PUBLIC_KEY = "pk-lf-xxxxxxxx"
LANGFUSE_SECRET_KEY = "sk-lf-xxxxxxxx"

_client = httpx.Client(
    base_url=LANGFUSE_HOST,
    auth=(LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY),
    verify=False,
    trust_env=False,
    timeout=httpx.Timeout(30.0, read=120.0),
)

def lf_get(path, **params):
    r = _client.get(path, params={k: v for k, v in params.items() if v is not None})
    r.raise_for_status()
    return r.json()

h = _client.get("/api/public/health"); h.raise_for_status()
print("Connected OK ->", LANGFUSE_HOST, "|", h.json())

## 2. Configuration — control what gets extracted & exported

**Control panel.** Three groups: trace/run-item fields, judge behaviour, circular extraction.

`FIELD_DEFINITIONS` doubles as documentation: each entry is `(include?, "definition")`.
The definition text is written verbatim into the Excel README sheet.

In [ ]:
# ============================================================
#  TRACE / RUN-ITEM FIELDS  — (include?, definition-for-README)
# ============================================================
FIELD_DEFINITIONS = {
    "run_name":        (True,  "Name of the experiment/run this row belongs to."),
    "dataset_item_id": (True,  "UUID of the dataset test case that was executed."),
    "input":           (True,  "What the application actually received (main trace input)."),
    "expected_output": (True,  "Gold/reference answer from the dataset item (ground truth)."),
    "actual_output":   (True,  "What the application actually produced (main trace output)."),
    "extracted_metadata": (True, "Circulars from the Vector Store Retriever step: timestamp | filename | attachment_id | summary, one per line."),
    "reasoning_chain": (False, "Concatenated outputs of the app's own intermediate steps (NOT the judge)."),
    "n_observations":  (False, "Count of application-side observations on the main trace."),
    "trace_name":      (False, "Name of the main trace."),
    "latency_ms":      (False, "End-to-end latency of the main trace, milliseconds."),
    "total_cost":      (False, "Total model cost recorded on the main trace."),
    "trace_id":        (True,  "UUID of the main trace (for cross-reference in the UI)."),
    "run_item_id":     (False, "UUID linking the dataset item to this run execution."),
    "observation_id":  (False, "Specific observation id, if the run item is scoped to one."),
}
FIELDS = {k: v[0] for k, v in FIELD_DEFINITIONS.items()}

# ============================================================
#  JUDGE / EVALUATOR  (single evaluator -> wide columns)
# ============================================================
JUDGE_INCLUDE         = True   # read trace.scores[] at all
JUDGE_FETCH_REASONING = True   # include score.comment (judge's reasoning)
JUDGE_FETCH_PROMPT    = True   # follow score -> execution trace -> judge input prompt + raw response

JUDGE_DEFINITIONS = {
    "score":          (True,  "The evaluator's score value (numeric/boolean) or category."),
    "reasoning":      (JUDGE_FETCH_REASONING, "The judge's written justification for the score (score.comment)."),
    "judge_prompt":   (JUDGE_FETCH_PROMPT,    "The exact prompt the judge LLM received (judge execution-trace input)."),
    "judge_response": (JUDGE_FETCH_PROMPT,    "The judge LLM's raw response (judge execution-trace output)."),
}

# ============================================================
#  CIRCULAR METADATA  (from the retriever observation)
# ============================================================
EXTRACT_CIRCULAR_METADATA = True
RETRIEVER_OBS_NAME = "Vector Store Retriever (Detailed)"  # observation 'name' to match
RETRIEVER_OBS_TYPE = "retriever"   # also require this 'type'; set None to match on name only
CIRCULAR_DOCS_KEY  = "documents"   # key under observation.output holding the array
CIRCULAR_FIELDS    = ["timestamp", "filename", "attachment_id", "summary"]  # per-document fields

# ============================================================
#  GENERAL
# ============================================================
TEXT_LIMIT = 8000   # truncate long cells for Excel safety (None = no truncation)

# circular extraction needs app-side observations regardless of the reasoning_chain toggle
INCLUDE_APP_OBSERVATIONS = bool(FIELDS.get("reasoning_chain") or FIELDS.get("n_observations"))
_NEED_OBS = INCLUDE_APP_OBSERVATIONS or EXTRACT_CIRCULAR_METADATA

print("Active fields :", [k for k, v in FIELDS.items() if v])
print("Judge         :", {k: v[0] for k, v in JUDGE_DEFINITIONS.items()})
print("Circulars     :", EXTRACT_CIRCULAR_METADATA, "->", CIRCULAR_FIELDS)
print("Fetch app obs :", _NEED_OBS)

## 3. Imports & helpers

In [ ]:
import json, datetime as dt
from pathlib import Path
from urllib.parse import quote
import pandas as pd

OUT_DIR = Path("langfuse_exports"); OUT_DIR.mkdir(exist_ok=True)
def _ts(): return dt.datetime.now().strftime("%Y%m%d_%H%M%S")

def _flatten(value, max_len=None):
    if value is None: return None
    if isinstance(value, (dict, list)):
        value = json.dumps(value, ensure_ascii=False, default=str)
    value = str(value)
    if max_len and len(value) > max_len:
        value = value[:max_len] + " ...[truncated]"
    return value

def _as_obj(v):
    """output/input may be a dict or a JSON string; return a Python object or None."""
    if v is None: return None
    if isinstance(v, (dict, list)): return v
    if isinstance(v, str):
        try: return json.loads(v)
        except Exception: return None
    return None

def paginate(path, data_key="data", **params):
    page, out = 1, []
    while True:
        js = lf_get(path, page=page, limit=100, **params)
        data = js.get(data_key, []) or []
        if not data: break
        out.extend(data)
        meta = js.get("meta") or {}
        tp = meta.get("totalPages")
        if tp is not None and page >= tp: break
        if tp is None and len(data) < 100: break
        page += 1
    return out

def _apply_fields(row):
    return {k: row.get(k) for k, v in FIELDS.items() if v}

print("Helpers ready. Exports ->", OUT_DIR.resolve())

## 4. List all datasets

In [ ]:
datasets = paginate("/api/public/v2/datasets")
df_datasets = pd.DataFrame([{
    "name": d.get("name"), "id": d.get("id"),
    "description": d.get("description"),
    "metadata": _flatten(d.get("metadata")),
    "createdAt": d.get("createdAt"), "updatedAt": d.get("updatedAt"),
} for d in datasets])
print(f"{len(df_datasets)} dataset(s)\n")
df_datasets

## 5. Pick a dataset

In [ ]:
DATASET_NAME = "your-dataset-name"   # <-- edit me
dataset = lf_get(f"/api/public/v2/datasets/{quote(DATASET_NAME, safe='')}")
print("Dataset:", dataset.get("name"), "| id:", dataset.get("id"))

## 6. Mode A — Dataset items only
Filtered by `datasetName` (the `datasetId` filter is silently ignored upstream).

In [ ]:
items = paginate("/api/public/dataset-items", datasetName=DATASET_NAME)
df_items = pd.DataFrame([{
    "item_id": it.get("id"), "status": it.get("status"),
    "input": _flatten(it.get("input")),
    "expected_output": _flatten(it.get("expectedOutput")),
    "metadata": _flatten(it.get("metadata")),
    "source_trace_id": it.get("sourceTraceId"),
    "createdAt": it.get("createdAt"),
} for it in items])
print(f"{len(df_items)} item(s)")
df_items.head(20)

## 7. List runs for this dataset

In [ ]:
runs = paginate(f"/api/public/datasets/{quote(DATASET_NAME, safe='')}/runs")
df_runs = pd.DataFrame([{
    "run_name": r.get("name"), "run_id": r.get("id"),
    "description": r.get("description"),
    "metadata": _flatten(r.get("metadata")),
    "createdAt": r.get("createdAt"), "updatedAt": r.get("updatedAt"),
} for r in runs])
print(f"{len(df_runs)} run(s)")
df_runs

## 8. Mode B — Runs + outputs + judge + circulars

Per run item: main trace (input/actual), dataset item (expected), judge scores
(value + reasoning + judge prompt/response), and circulars from the retriever step.

In [ ]:
SELECTED_RUNS = []   # [] = all runs, or e.g. ["baseline-v1"]

target = df_runs["run_name"].tolist() if not SELECTED_RUNS else SELECTED_RUNS
print(f"Pulling {len(target)} run(s)...")

_trace_cache, _obs_cache, _item_cache = {}, {}, {}

def get_run_with_items(run_name):
    return lf_get(f"/api/public/datasets/{quote(DATASET_NAME, safe='')}/runs/{quote(run_name, safe='')}")

def get_trace(trace_id):
    if not trace_id: return None
    if trace_id not in _trace_cache:
        try: _trace_cache[trace_id] = lf_get(f"/api/public/traces/{trace_id}")
        except Exception as e:
            _trace_cache[trace_id] = None; print(f"  ! trace {trace_id}: {e}")
    return _trace_cache[trace_id]

def get_dataset_item(item_id):
    if not item_id: return None
    if item_id not in _item_cache:
        try: _item_cache[item_id] = lf_get(f"/api/public/dataset-items/{item_id}")
        except Exception: _item_cache[item_id] = None
    return _item_cache[item_id]

def get_app_observations(trace_id):
    if not trace_id: return []
    if trace_id not in _obs_cache:
        try: _obs_cache[trace_id] = paginate("/api/public/observations", traceId=trace_id)
        except Exception: _obs_cache[trace_id] = []
    return _obs_cache[trace_id]

def _score_execution_trace_id(score):
    for k in ("executionTraceId", "evalExecutionTraceId", "jobExecutionId", "executionId"):
        if score.get(k): return score.get(k)
    md_ = score.get("metadata") or {}
    if isinstance(md_, dict):
        for k in ("executionTraceId", "traceId", "execution_trace_id"):
            if md_.get(k): return md_.get(k)
    return None

def build_judge_detail(score):
    d = {"name": score.get("name"), "value": score.get("value"),
         "string_value": score.get("stringValue"), "data_type": score.get("dataType"),
         "reasoning": score.get("comment") if JUDGE_FETCH_REASONING else None,
         "judge_prompt": None, "judge_response": None, "execution_trace_id": None}
    if JUDGE_FETCH_PROMPT:
        exec_id = _score_execution_trace_id(score)
        d["execution_trace_id"] = exec_id
        if exec_id:
            et = get_trace(exec_id)
            if et:
                d["judge_prompt"]   = et.get("input")
                d["judge_response"] = et.get("output")
    return d

def extract_circulars(observations):
    """From retriever observations, pull output.documents[*] -> CIRCULAR_FIELDS."""
    found = []
    for o in observations:
        if o.get("name") != RETRIEVER_OBS_NAME:
            continue
        if RETRIEVER_OBS_TYPE and (o.get("type") or "").lower() != RETRIEVER_OBS_TYPE.lower():
            continue
        out = _as_obj(o.get("output"))
        docs = (out or {}).get(CIRCULAR_DOCS_KEY) if isinstance(out, dict) else None
        if not isinstance(docs, list):
            continue
        for doc in docs:
            if not isinstance(doc, dict): continue
            found.append({f: doc.get(f) for f in CIRCULAR_FIELDS})
    return found

def circulars_to_cell(circs):
    """Human-readable wrapped cell: one circular per line, fields ' | ' separated."""
    if not circs: return None
    lines = []
    for c in circs:
        lines.append(" | ".join("" if c.get(f) is None else str(c.get(f)) for f in CIRCULAR_FIELDS))
    return "\n".join(lines)

flat_rows, nested = [], []

for run_name in target:
    run_full = get_run_with_items(run_name)
    run_items = run_full.get("datasetRunItems") or run_full.get("data") or []
    print(f"  run '{run_name}': {len(run_items)} item(s)")
    run_node = {"run": {k: v for k, v in run_full.items()
                        if k not in ("datasetRunItems", "data")}, "items": []}

    for it in run_items:
        trace_id = it.get("traceId")
        item_id  = it.get("datasetItemId")
        di = get_dataset_item(item_id)
        expected = di.get("expectedOutput") if di else None
        trace = get_trace(trace_id)

        observations = get_app_observations(trace_id) if _NEED_OBS else []

        # app-side reasoning chain (optional)
        reasoning = None
        if FIELDS.get("reasoning_chain"):
            parts = [f"[{o.get('type','')}:{o.get('name','')}] {_flatten(o.get('output'),2000)}"
                     for o in observations if o.get("output") is not None]
            reasoning = "\n".join(parts) if parts else None

        # circulars
        circs = extract_circulars(observations) if EXTRACT_CIRCULAR_METADATA else []

        # judge
        scores = (trace.get("scores") if trace else None) or []
        judge_details = [build_judge_detail(s) for s in scores] if JUDGE_INCLUDE else []

        base = {
            "run_name": run_name,
            "dataset_item_id": item_id,
            "input": _flatten(trace.get("input") if trace else None, TEXT_LIMIT),
            "expected_output": _flatten(expected, TEXT_LIMIT),
            "actual_output": _flatten(trace.get("output") if trace else None, TEXT_LIMIT),
            "extracted_metadata": circulars_to_cell(circs),
            "reasoning_chain": _flatten(reasoning, TEXT_LIMIT),
            "n_observations": len(observations) if _NEED_OBS else None,
            "trace_name": trace.get("name") if trace else None,
            "latency_ms": trace.get("latency") if trace else None,
            "total_cost": trace.get("totalCost") if trace else None,
            "trace_id": trace_id,
            "run_item_id": it.get("id"),
            "observation_id": it.get("observationId"),
        }
        row = _apply_fields(base)

        # judge -> wide columns (single evaluator, but loops safely if more appear)
        if JUDGE_INCLUDE:
            for jd in judge_details:
                nm = jd["name"] or "score"
                suffix = "" if len(judge_details) == 1 else f"__{nm}"
                val = jd["value"] if jd["value"] is not None else jd["string_value"]
                if JUDGE_DEFINITIONS["score"][0]:
                    row[f"score{suffix}"] = val
                if JUDGE_DEFINITIONS["reasoning"][0]:
                    row[f"reasoning{suffix}"] = _flatten(jd["reasoning"], TEXT_LIMIT)
                if JUDGE_DEFINITIONS["judge_prompt"][0]:
                    row[f"judge_prompt{suffix}"]   = _flatten(jd["judge_prompt"], TEXT_LIMIT)
                    row[f"judge_response{suffix}"] = _flatten(jd["judge_response"], TEXT_LIMIT)

        flat_rows.append(row)
        run_node["items"].append({
            "run_item": it, "dataset_item": di, "trace": trace,
            "observations": observations, "circulars": circs,
            "scores": scores, "judge_details": judge_details,
        })
    nested.append(run_node)

df_run_items = pd.DataFrame(flat_rows)
print(f"\nTotal run items: {len(df_run_items)}")
print("Columns:", list(df_run_items.columns))
df_run_items.head(20)

## 9. Inspect one full record

In [ ]:
if not df_run_items.empty:
    for k, v in df_run_items.iloc[0].to_dict().items():
        s = str(v)
        print(f"--- {k} ---\n{s[:1500]}{' ...[more]' if len(s) > 1500 else ''}\n")
else:
    print("No run items - run cell 8 first.")

## 10. Export — styled Excel + flat JSON + nested JSON

- **Excel**: README sheet (definitions) + datasets / items / runs / **run_items** (styled:
  ordered columns, bold frozen header, autofilter, wrapped text, tuned widths).
- **Flat JSON**: same selected fields, one record per row, with a config header.
- **Nested JSON**: full raw archive (all document fields, full traces, all scores) — ignores
  field toggles so nothing is lost.

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

EXPORT_DATASETS, EXPORT_ITEMS, EXPORT_RUNS, EXPORT_RUN_ITEMS, EXPORT_NESTED_JSON = True, True, True, True, True

stamp = _ts(); safe = quote(DATASET_NAME, safe="")

# ---- desired wide column order for run_items (judge cols inserted dynamically) ----
def ordered_run_item_columns(df):
    lead = [c for c in ["run_name", "dataset_item_id",
                        "input", "expected_output", "actual_output",
                        "extracted_metadata"] if c in df.columns]
    judge = [c for c in df.columns if c.startswith(("score", "reasoning", "judge_prompt", "judge_response"))
             and c != "reasoning_chain"]
    tail = [c for c in ["reasoning_chain", "n_observations", "trace_name",
                        "latency_ms", "total_cost", "trace_id", "run_item_id",
                        "observation_id"] if c in df.columns]
    seen = set(lead + judge + tail)
    rest = [c for c in df.columns if c not in seen]
    return lead + judge + rest + tail

# columns that hold long text -> wrap + wide
WIDE_WRAP = {"input", "expected_output", "actual_output", "extracted_metadata",
             "reasoning", "reasoning_chain", "judge_prompt", "judge_response"}
NARROW    = {"run_name", "dataset_item_id", "trace_id", "run_item_id",
             "observation_id", "score", "latency_ms", "total_cost", "n_observations"}

HEADER_FILL = PatternFill("solid", fgColor="2F5597")
HEADER_FONT = Font(bold=True, color="FFFFFF", size=11)
THIN = Side(style="thin", color="D9D9D9")
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
TOPLEFT = Alignment(horizontal="left", vertical="top", wrap_text=True)

def style_sheet(ws, wrap_cols=None, freeze=True):
    wrap_cols = wrap_cols or set()
    # header
    for cell in ws[1]:
        cell.fill = HEADER_FILL; cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = BORDER
    # widths + body alignment
    headers = [c.value for c in ws[1]]
    for idx, name in enumerate(headers, start=1):
        col = get_column_letter(idx)
        if name in wrap_cols:        width = 60
        elif name in NARROW:         width = 18
        elif str(name).startswith(("judge_prompt", "judge_response", "reasoning")): width = 60
        else:                        width = 28
        ws.column_dimensions[col].width = width
    for r in range(2, ws.max_row + 1):
        for idx, name in enumerate(headers, start=1):
            cell = ws.cell(row=r, column=idx)
            cell.border = BORDER
            cell.alignment = TOPLEFT if name in wrap_cols else Alignment(horizontal="left", vertical="top", wrap_text=False)
    if ws.max_row >= 1:
        ws.auto_filter.ref = f"A1:{get_column_letter(ws.max_column)}{ws.max_row}"
    if freeze:
        ws.freeze_panes = "A2"

def build_readme_df():
    rows = [("SHEET: run_items", "")]
    for k, (inc, desc) in FIELD_DEFINITIONS.items():
        if inc: rows.append((k, desc))
    for k, (inc, desc) in JUDGE_DEFINITIONS.items():
        if inc: rows.append((k + " (and __<evaluator> variants if >1)", desc))
    rows.append(("", ""))
    rows.append(("extracted_metadata format", "Each line: " + " | ".join(CIRCULAR_FIELDS)))
    rows.append(("source observation", f'name="{RETRIEVER_OBS_NAME}", type="{RETRIEVER_OBS_TYPE}", output.{CIRCULAR_DOCS_KEY}[*]'))
    rows.append(("", ""))
    rows.append(("OTHER SHEETS", ""))
    rows.append(("datasets", "All datasets in the project."))
    rows.append(("items", "Dataset test cases (Mode A): input, expected_output, metadata."))
    rows.append(("runs", "Experiments/runs for the selected dataset."))
    return pd.DataFrame(rows, columns=["Field / Sheet", "Definition"])

xlsx_path = OUT_DIR / f"{safe}_{stamp}.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xl:
    build_readme_df().to_excel(xl, sheet_name="README", index=False)
    if EXPORT_DATASETS and not df_datasets.empty:
        df_datasets.to_excel(xl, sheet_name="datasets", index=False)
    if EXPORT_ITEMS and 'df_items' in dir() and not df_items.empty:
        df_items.to_excel(xl, sheet_name="items", index=False)
    if EXPORT_RUNS and 'df_runs' in dir() and not df_runs.empty:
        df_runs.to_excel(xl, sheet_name="runs", index=False)
    if EXPORT_RUN_ITEMS and 'df_run_items' in dir() and not df_run_items.empty:
        df_ri = df_run_items[ordered_run_item_columns(df_run_items)]
        df_ri.to_excel(xl, sheet_name="run_items", index=False)

    wb = xl.book
    # README styling
    rm = wb["README"]
    for cell in rm[1]:
        cell.fill = HEADER_FILL; cell.font = HEADER_FONT
    rm.column_dimensions["A"].width = 42; rm.column_dimensions["B"].width = 90
    for r in range(2, rm.max_row + 1):
        rm.cell(row=r, column=1).font = Font(bold=True)
        rm.cell(row=r, column=2).alignment = Alignment(wrap_text=True, vertical="top")
    rm.freeze_panes = "A2"
    # data sheets
    for name in ("datasets", "items", "runs"):
        if name in wb.sheetnames: style_sheet(wb[name], wrap_cols={"metadata", "description", "input", "expected_output"})
    if "run_items" in wb.sheetnames:
        style_sheet(wb["run_items"], wrap_cols=WIDE_WRAP)

print("Excel  ->", xlsx_path)

# ---- flat JSON ----
payload = {"exported_at": stamp, "host": LANGFUSE_HOST, "dataset_name": DATASET_NAME,
           "field_selection": FIELDS,
           "judge_config": {k: v[0] for k, v in JUDGE_DEFINITIONS.items()},
           "circular_config": {"enabled": EXTRACT_CIRCULAR_METADATA, "obs_name": RETRIEVER_OBS_NAME,
                               "fields": CIRCULAR_FIELDS}}
if EXPORT_ITEMS and 'df_items' in dir():    payload["items"] = df_items.to_dict(orient="records")
if EXPORT_RUNS and 'df_runs' in dir():       payload["runs"] = df_runs.to_dict(orient="records")
if EXPORT_RUN_ITEMS and 'df_run_items' in dir(): payload["run_items_flat"] = df_run_items.to_dict(orient="records")
json_path = OUT_DIR / f"{safe}_{stamp}.json"
json_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str))
print("JSON   ->", json_path)

# ---- nested JSON (raw archive) ----
if EXPORT_NESTED_JSON and 'nested' in dir() and nested:
    nested_path = OUT_DIR / f"{safe}_{stamp}_nested.json"
    nested_path.write_text(json.dumps(nested, ensure_ascii=False, indent=2, default=str))
    print("Nested ->", nested_path)
print("\nDone.")

---
### Notes
- **Circulars** come from `output.documents[*]` of the observation named
  *"Vector Store Retriever (Detailed)"* (type `retriever`). Each line in `extracted_metadata`
  is `timestamp | filename | attachment_id | summary`. If a run has no such step, the cell is blank.
- **Judge prompt** needs platform >= 3.119.0 (execution-trace tracing). If `judge_prompt` is
  empty but you see judge traces in the UI, print `nested[0]["items"][0]["scores"][0].keys()`
  and tell me the key — I'll pin the execution-trace id field.
- **Single evaluator** -> judge columns are `score`, `reasoning`, `judge_prompt`,
  `judge_response` (no suffix). If a second evaluator ever appears, columns auto-suffix with
  `__<evaluator-name>`.
- Excel long-text columns are wrapped & width-tuned; header is frozen with an autofilter.
  Full untruncated content always lives in the JSON exports.
- `datasetId` item filter is broken upstream -> we use `datasetName`.